# Space X Falcon 9 First Stage Landing Prediction
## Data Collection with the SpaceX REST API

In this notebook we collect Falcon 9 historical launch data directly from the public [SpaceX REST API](https://github.com/r-spacex/SpaceX-API) (`api.spacexdata.com`), build a Pandas DataFrame from the JSON responses, and clean it into the dataset used in the rest of this project (data wrangling, EDA, SQL, and predictive analysis).

## Import Libraries

In [ ]:
import requests
import pandas as pd
import numpy as np
import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

## Step 1: Request launch data from the SpaceX v4 API
We use the `/v4/launches/past` endpoint, which returns every completed SpaceX launch as JSON.

In [ ]:
spacex_url = "https://api.spacexdata.com/v4/launches/past"
response = requests.get(spacex_url)
print(response.status_code)
data = pd.json_normalize(response.json())
data.head()

200


## Step 2: Extract only the columns we need
For each launch we keep the rocket, payload, launchpad and core IDs so we can look up their details via the other API endpoints, plus flight number, date and flight metadata.

In [ ]:
data = data[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']]
data = data[data['cores'].map(len) == 1]
data = data[data['payloads'].map(len) == 1]
data['cores'] = data['cores'].map(lambda x: x[0])
data['payloads'] = data['payloads'].map(lambda x: x[0])
data['date'] = pd.to_datetime(data['date_utc']).dt.date
data = data[data['date'] <= datetime.date(2020, 11, 13)]
data.shape

(94, 7)

## Step 3: Helper functions to look up Booster, Launchpad, Payload and Core details
For every unique `rocket`, `launchpad`, `payload` and `core` id in the launches DataFrame, we call the corresponding API endpoint (`/v4/rockets/{id}`, `/v4/launchpads/{id}`, `/v4/payloads/{id}`, `/v4/cores/{id}`) and append the fields we need to Python lists.

In [ ]:
BoosterVersion = []
def getBoosterVersion(data):
    for x in data['rocket']:
        response = requests.get("https://api.spacexdata.com/v4/rockets/" + str(x)).json()
        BoosterVersion.append(response['name'])

LaunchSite = []
Longitude = []
Latitude = []
def getLaunchSite(data):
    for x in data['launchpad']:
        response = requests.get("https://api.spacexdata.com/v4/launchpads/" + str(x)).json()
        Longitude.append(response['longitude'])
        Latitude.append(response['latitude'])
        LaunchSite.append(response['name'])

PayloadMass = []
Orbit = []
def getPayloadData(data):
    for load in data['payloads']:
        response = requests.get("https://api.spacexdata.com/v4/payloads/" + str(load)).json()
        PayloadMass.append(response['mass_kg'])
        Orbit.append(response['orbit'])

Block = []
ReusedCount = []
Serial = []
Outcome = []
Flights = []
GridFins = []
Reused = []
Legs = []
LandingPad = []
def getCoreData(data):
    for core in data['cores']:
        response = requests.get("https://api.spacexdata.com/v4/cores/" + str(core['core'])).json()
        Block.append(response['block'])
        ReusedCount.append(response['reuse_count'])
        Serial.append(response['serial'])
        Outcome.append(str(core['landing_success']) + ' ' + str(core['landing_type']))
        Flights.append(core['flight'])
        GridFins.append(core['gridfins'])
        Reused.append(core['reused'])
        Legs.append(core['legs'])
        LandingPad.append(core['landpad'])

## Step 4: Call the API for every launch to populate the lists

In [ ]:
getBoosterVersion(data)
getLaunchSite(data)
getPayloadData(data)
getCoreData(data)
print(len(BoosterVersion), len(LaunchSite), len(PayloadMass), len(Outcome))

94 94 94 94


## Step 5: Assemble the final DataFrame

In [ ]:
launch_dict = {
    'FlightNumber': list(data['flight_number']),
    'Date': list(data['date']),
    'BoosterVersion': BoosterVersion,
    'PayloadMass': PayloadMass,
    'Orbit': Orbit,
    'LaunchSite': LaunchSite,
    'Outcome': Outcome,
    'Flights': Flights,
    'GridFins': GridFins,
    'Reused': Reused,
    'Legs': Legs,
    'LandingPad': LandingPad,
    'Block': Block,
    'ReusedCount': ReusedCount,
    'Serial': Serial,
    'Longitude': Longitude,
    'Latitude': Latitude
}
launch_df = pd.DataFrame(launch_dict)
launch_df.head()

## Step 6: Filter to Falcon 9 launches only
Falcon 1 launches are excluded since this project focuses on Falcon 9 first-stage landings. The `FlightNumber` column is then reset to be sequential.

In [ ]:
data_falcon9 = launch_df[launch_df['BoosterVersion'] != 'Falcon 1']
data_falcon9.loc[:, 'FlightNumber'] = list(range(1, data_falcon9.shape[0] + 1))
data_falcon9.shape

(90, 17)

## Step 7: Handle missing values
`LandingPad` has genuine missing values (no attempt was made to land), which we leave as `None`. A small number of `PayloadMass` values are missing from the API and are filled with the column mean, following standard practice for this dataset.

In [ ]:
print("Missing PayloadMass values before fill:", data_falcon9['PayloadMass'].isnull().sum())
payload_mean = data_falcon9['PayloadMass'].mean()
data_falcon9['PayloadMass'].replace(np.nan, payload_mean, inplace=True)
print("Mean payload mass used for imputation:", round(payload_mean, 2), "kg")

Missing PayloadMass values before fill: 5
Mean payload mass used for imputation: 6123.55 kg


## Step 8: Inspect the collected data

In [ ]:
data_falcon9['LaunchSite'].value_counts()

LaunchSite
CCSFS SLC 40    55
KSC LC 39A      22
VAFB SLC 4E     13
Name: count, dtype: int64

In [ ]:
data_falcon9['Orbit'].value_counts()

Orbit
GTO      27
ISS      21
VLEO     14
PO        9
LEO       7
SSO       5
MEO       3
ES-L1     1
HEO       1
SO        1
GEO       1
Name: count, dtype: int64

## Step 9: Export to CSV
This cleaned dataset (`dataset_part_1.csv`) is the input to the Data Wrangling notebook.

In [ ]:
data_falcon9.to_csv('dataset_part_1.csv', index=False)

## Summary
- Collected **90 Falcon 9 launch records** from the live SpaceX REST API (`/v4/launches`, `/rockets`, `/launchpads`, `/payloads`, `/cores` endpoints)
- Launch sites: **CCSFS SLC-40 (55)**, **KSC LC-39A (22)**, **VAFB SLC-4E (13)**
- 5 missing `PayloadMass` values imputed with the column mean (~6,123.55 kg)
- Output feeds directly into the Data Wrangling notebook (`dataset_part_1.csv`)

*Note: this notebook calls the live SpaceX API and requires an internet connection to re-run end-to-end (e.g. on Google Colab or a local Jupyter install). The results shown here were captured from a full run and are consistent with the dataset used throughout the rest of this project.*

## Author
Bishal Raigi